<a href="https://colab.research.google.com/github/eenoda/Dungeon-of-Demon/blob/main/CBE_Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# 셀 1: 환경 설정 및 라이브러리 임포트
# ==============================================================================
print(">>> [Cell 1] Setting up environment and importing libraries...")

import os
import sys
import json
import time
import glob
import math
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from tqdm.auto import tqdm

# --- Google Drive 마운트 ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("Google Drive mounted successfully.")
except ImportError:
    print("Not in a Google Colab environment. Skipping drive mount.")
    IN_COLAB = False

# --- 프로젝트 경로 설정 및 모듈 임포트 ---
# config.py에 정의된 경로를 사용하기 위해 경로 추가
# 이 셀을 실행하는 위치가 /content/ 라고 가정
PROJECT_ROOT = '/content/drive/MyDrive/CBE-FGC-Project'
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    print(f"Added '{PROJECT_ROOT}' to system path.")

# 이제 우리 모듈을 임포트할 수 있습니다.
from config import ProjectConfig
from data.dataset import MultiTaskDataset
from data.collate import MultiTaskCollator
from model.cbe_fgc import CBE_FGC_Model
from trainer.trainer import train_one_epoch, evaluate

print("\n>>> Environment setup complete.")

>>> [Cell 1] Setting up environment and importing libraries...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully.
Added '/content/drive/MyDrive/CBE-FGC-Project' to system path.

>>> Environment setup complete.


In [2]:
# ==============================================================================
# 셀 2: 설정(Config) 로드 및 시드 고정
# ==============================================================================
print(">>> [Cell 2] Loading configuration and setting random seed...")

# --- 설정 불러오기 ---
config = ProjectConfig()

# --- 이번 실험을 위한 설정 수정 ---
config.data.use_large_corpus = True # 대규모 코퍼스 사용 시 주석 해제
config.train.batch_size = 32        # GPU 메모리에 따라 조정
# config.train.epochs = 5           # 테스트를 위해 에폭 수 줄이기

# --- 재현성을 위한 시드 고정 함수 ---
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(config.train.seed)

# --- 출력 디렉토리 생성 ---
os.makedirs(config.train.output_dir, exist_ok=True)

# --- 장치 설정 ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Using large corpus: {config.data.use_large_corpus}")
print(f"Batch size: {config.train.batch_size}")

print("\n>>> Configuration loaded and seed fixed.")

>>> [Cell 2] Loading configuration and setting random seed...
Using device: cuda
Using large corpus: True
Batch size: 32

>>> Configuration loaded and seed fixed.


In [3]:
# ==============================================================================
# 셀 3: 토크나이저 및 데이터 준비
# ==============================================================================
print(">>> [Cell 3] Preparing tokenizer and datasets...")

# --- 토크나이저 로드 및 스페셜 토큰 추가 ---
tokenizer = AutoTokenizer.from_pretrained(config.data.tokenizer_name)
special_tokens_dict = {'additional_special_tokens': ['[QUERY]']}
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)
print(f"Added {num_added_toks} special token: '[QUERY]'")

# --- 데이터셋 및 데이터로더 생성 ---
corpus_path = config.data.get_corpus_path()
print(f"Loading data from: {corpus_path}")

# 전체 데이터셋을 한번에 로드
full_dataset = MultiTaskDataset(
    file_path=corpus_path,
    tokenizer=tokenizer,
    preproc_config=config.preprocessing,
    data_config=config.data
)

# Train/Validation 분리 (90% / 10%)
train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

print(f"Train dataset size: {len(train_dataset):,}")
print(f"Validation dataset size: {len(val_dataset):,}")

# Collator 인스턴스화
collator = MultiTaskCollator(pad_token_id=tokenizer.pad_token_id)

# 데이터로더 생성
train_dataloader = DataLoader(
    train_dataset,
    batch_size=config.train.batch_size,
    shuffle=True,
    collate_fn=collator,
    num_workers=2,
    pin_memory=True
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=config.train.batch_size,
    shuffle=False,
    collate_fn=collator,
    num_workers=2,
    pin_memory=True
)

print("\n>>> Tokenizer and data loaders are ready.")

>>> [Cell 3] Preparing tokenizer and datasets...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Added 1 special token: '[QUERY]'
Loading data from: /content/drive/MyDrive/CBE-FGC-Project/data/copus for PoC_large.pt
INFO: Loading dataset from /content/drive/MyDrive/CBE-FGC-Project/data/copus for PoC_large.pt...
INFO: Dataset loaded. Total sentences: 290,719
Train dataset size: 261,647
Validation dataset size: 29,072

>>> Tokenizer and data loaders are ready.


In [4]:
# ==============================================================================
# 셀 4: 모델 및 옵티마이저 초기화
# ==============================================================================
print(">>> [Cell 4] Initializing model, optimizer, and scheduler...")

# --- 모델 설정 업데이트 ---
config.model.vocab_size = len(tokenizer)
config.model.pad_token_id = tokenizer.pad_token_id

# --- 모델 초기화 ---
model = CBE_FGC_Model(config.model).to(device)
model.token_embedding.weight.data.normal_(mean=0.0, std=0.02) # 임베딩 가중치 초기화

# 모델의 토큰 임베딩 크기를 토크나이저에 맞춰 리사이즈 (필수는 아니지만 안전장치)
# model.resize_token_embeddings(len(tokenizer)) # Embedding 레이어를 새로 만들었으므로 필요 없음

print(f"Model initialized on {device}.")
print(f"Total parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# --- 옵티마이저 및 스케줄러 설정 ---
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.train.learning_rate,
    weight_decay=config.train.weight_decay
)

num_training_steps = len(train_dataloader) * config.train.epochs
num_warmup_steps = int(num_training_steps * config.train.warmup_ratio)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

print(f"Optimizer and scheduler are set up.")
print(f"Total training steps: {num_training_steps:,}")
print(f"Warmup steps: {num_warmup_steps:,}")

>>> [Cell 4] Initializing model, optimizer, and scheduler...
Model initialized on cuda.
Total parameters: 19,844,101
Optimizer and scheduler are set up.
Total training steps: 81,770
Warmup steps: 8,177


In [5]:
from google.colab import userdata
key=userdata.get('wandb')

In [ ]:
# ==============================================================================
# Final Integrated Training & Evaluation Cell
#
# Features:
# - Weights & Biases (W&B) for real-time logging and visualization.
# - torch.compile for significant speedup on PyTorch 2.0+.
# - Step-wise checkpointing for safety during long training runs.
# - Robust resume logic to continue from the latest state.
# - Seamless integration with the modular `trainer.py`.
# ==============================================================================
print(">>> Initializing Final Integrated Training Cell...")

# --- 1. Import necessary libraries ---
import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler
from torch.utils.data import DataLoader, random_split
import time
import os
import json
import glob

# (Optional but Recommended) Weights & Biases for logging
try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    WANDB_AVAILABLE = False
    print("wandb not installed. Skipping W&B logging. To enable, run: !pip install wandb")


# --- 2. Load Project-Specific Modules ---
# 이 셀을 실행하기 전에 아래 파일들이 올바른 경로에 정의되어 있어야 합니다.
# from config import ProjectConfig
# from model.cbe_fgc import CBE_FGC_Model
# from data.dataset import YourDataset, create_dataloaders # 데이터셋 로직은 별도 파일로 가정
from trainer.trainer import train_one_epoch, evaluate, save_checkpoint # 이전에 정의한 trainer 함수들

# --- [Placeholder] 아래는 실제 프로젝트 파일들을 임포트해야 합니다 ---
# 여기서는 시연을 위해 config와 trainer 함수들이 이 셀 안에 있다고 가정합니다.
# 실제 사용 시에는 이 부분을 주석 처리하고, 위 from ... import ... 구문을 사용하세요.
config = ProjectConfig()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# --- End of Placeholder ---


# --- 3. Initialize Weights & Biases (if available) ---
if WANDB_AVAILABLE:
    try:
        wandb.login() # First time in Colab might require authentication

        # Unique run name for each experiment
        run_name = f"cbe_fgc_{time.strftime('%Y%m%d-%H%M%S')}"

        wandb.init(
            project="CBE-FGC-Project", # Your project name
            name=run_name,
            config={
                "model": config.model.__dict__,
                "train": config.train.__dict__,
                "data": config.data.__dict__,
                "preprocessing": config.preprocessing.__dict__,
            }
        )
        print(f"✅ W&B logging is enabled for run: {run_name}")
    except Exception as e:
        print(f"W&B initialization failed: {e}. Disabling W&B.")
        WANDB_AVAILABLE = False


# --- 4. Prepare Model, Optimizer, and DataLoaders ---
# 이 부분은 사용자의 기존 코드에 따라 설정됩니다.
# (예시)
# model = CBE_FGC_Model(config.model).to(device)
# train_dataloader, val_dataloader = create_dataloaders(...)
# optimizer = torch.optim.AdamW(...)
# scheduler = get_linear_schedule_with_warmup(...)
# --- (예시 종료) ---


# --- 5. Setup for Training Loop ---
print(">>> Preparing for main training loop...")
scaler = GradScaler(enabled=config.train.use_amp)
best_val_ppl = float("inf")
start_epoch = 0
resumed_step_in_epoch = 0 # To track step to resume from within an epoch

checkpoint_dir = config.train.output_dir
os.makedirs(checkpoint_dir, exist_ok=True)
latest_checkpoint_path = os.path.join(checkpoint_dir, "latest_checkpoint.pth")


# --- 6. Robust Checkpoint Loading Logic ---
if os.path.exists(latest_checkpoint_path):
    print(f"Resuming training from latest checkpoint: {latest_checkpoint_path}")
    checkpoint = torch.load(latest_checkpoint_path, map_location=device)

    # Handle state dict keys if model was compiled
    def unwrap_state_dict(state_dict):
        unwrapped = {}
        for key, value in state_dict.items():
            if key.startswith('_orig_mod.'):
                unwrapped[key[len('_orig_mod.'):]] = value
            else:
                unwrapped[key] = value
        return unwrapped

    model.load_state_dict(unwrap_state_dict(checkpoint['model_state_dict']))

    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])

    start_epoch = checkpoint.get('epoch', 0)
    resumed_step_in_epoch = checkpoint.get('step', 0)
    best_val_ppl = checkpoint.get('best_val_ppl', float('inf'))

    print(f"✅ Resumed from Epoch {start_epoch}, planning to start at Step {resumed_step_in_epoch}. Best PPL so far: {best_val_ppl:.4f}")

    # If we finished an epoch, start the next one from step 0
    if resumed_step_in_epoch >= len(train_dataloader):
        resumed_step_in_epoch = 0
        start_epoch += 1 # We completed the epoch, so we start the next one.
else:
    print("- No checkpoint found. Starting training from scratch.")


# --- 7. Apply torch.compile (after loading state_dict) ---
if torch.__version__ >= "2.0":
    print(">>> PyTorch 2.0+ detected. Preparing for compilation...")
    try:
        # 1. 컴파일을 유도하기 위한 아주 작은 '더미 배치'를 만듭니다.
        #    배치 사이즈를 1 또는 2로, 시퀀스 길이는 실제와 같게 설정합니다.
        print("Creating a small dummy batch to trigger compilation...")
        dummy_batch_size = 2
        dummy_input_ids = torch.randint(0, config.model.vocab_size,
                                        (dummy_batch_size, config.data.max_seq_len),
                                        device=device)
        dummy_attention_mask = torch.ones_like(dummy_input_ids)
        dummy_pos_ids = torch.zeros_like(dummy_input_ids)

        # 2. 모델을 컴파일합니다.
        model = torch.compile(model)
        print("Model wrapped with torch.compile. Now running a dummy forward pass to compile...")

        # 3. 더미 배치로 딱 한 번 forward pass를 실행하여 컴파일을 완료합니다.
        #    이 과정에서 RAM이 터지는지 확인합니다.
        with torch.no_grad(), autocast(enabled=config.train.use_amp):
             _ = model(
                 input_ids=dummy_input_ids,
                 attention_mask=dummy_attention_mask,
                 position_ids_for_recall=dummy_pos_ids
             )

        print("✅ Model compiled successfully using a small batch!")

    except Exception as e:
        print(f"⚠️ torch.compile failed: {e}. Running without compilation.")
else:
    print(">>> PyTorch version < 2.0. Skipping model compilation.")


# --- 8. The Main Training & Evaluation Loop ---
print("\n" + "="*50)
print("             STARTING TRAINING             ")
print("="*50 + "\n")

for epoch in range(start_epoch, config.train.epochs):
    epoch_start_time = time.time()

    # ------------------- TRAINING -------------------
    # Pass all necessary arguments to the trainer function
    avg_train_loss, avg_clm_loss, avg_qa_loss = train_one_epoch(
        model=model,
        dataloader=train_dataloader,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        device=device,
        qa_mlm_loss_weight=config.train.qa_mlm_loss_weight,
        current_epoch=epoch + 1,
        save_steps=config.train.save_steps,
        save_dir=checkpoint_dir,
        best_val_ppl=best_val_ppl, # Pass current best ppl for saving
        start_step=resumed_step_in_epoch, # Pass the step to resume from
        wandb_logging=WANDB_AVAILABLE
    )
    # After the first epoch, we always start from step 0
    resumed_step_in_epoch = 0

    # ------------------ EVALUATION ------------------
    avg_val_loss, avg_val_clm_loss, val_perplexity = evaluate(
        model=model,
        dataloader=val_dataloader,
        device=device,
        qa_mlm_loss_weight=config.train.qa_mlm_loss_weight
    )

    epoch_duration = time.time() - epoch_start_time

    # -------------- LOGGING & SAVING --------------
    print("\n" + "-"*20 + f" Epoch {epoch + 1} Summary " + "-"*20)
    print(f"  Duration: {epoch_duration:.2f}s")
    print(f"  Avg Train Loss -> Total: {avg_train_loss:.4f} (CLM: {avg_clm_loss:.4f}, QA: {avg_qa_loss:.4f})")
    print(f"  Avg Validation Loss -> Total: {avg_val_loss:.4f} (CLM: {avg_val_clm_loss:.4f})")
    print(f"  Validation Perplexity: {val_perplexity:.4f} (Best: {min(best_val_ppl, val_perplexity):.4f})")
    print("-" * (42 + len(str(epoch+1))))

    if WANDB_AVAILABLE:
        wandb.log({
            "epoch": epoch + 1,
            "avg_train_loss": avg_train_loss,
            "avg_train_clm_loss": avg_clm_loss,
            "avg_train_qa_loss": avg_qa_loss,
            "avg_val_loss": avg_val_loss,
            "avg_val_clm_loss": avg_val_clm_loss,
            "validation_perplexity": val_perplexity,
        })

    # Save best model
    is_best = val_perplexity < best_val_ppl
    if is_best:
        best_val_ppl = val_perplexity
        model_to_save = model._orig_mod if hasattr(model, '_orig_mod') else model
        best_model_path = os.path.join(checkpoint_dir, "best_model.pth")
        torch.save(model_to_save.state_dict(), best_model_path)
        print(f"  -> 🎉 New best model saved with PPL: {best_val_ppl:.4f} to {best_model_path}")

        with open(os.path.join(checkpoint_dir, "best_score.json"), "w") as f:
            json.dump({"best_val_ppl": best_val_ppl, "epoch": epoch + 1}, f)

        if WANDB_AVAILABLE:
            wandb.summary["best_validation_perplexity"] = best_val_ppl

    # Save latest checkpoint at the end of each epoch
    save_checkpoint(
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        epoch=epoch + 1, # Save as the *next* epoch to start from
        step=len(train_dataloader), # Indicate that this epoch is complete
        save_dir=checkpoint_dir,
        best_val_ppl=best_val_ppl
    )
    print(f"  -> Epoch {epoch + 1} final checkpoint saved.")


# --- 9. Finalize Training ---
print("\n\n" + "="*50)
print("           🎉 TRAINING COMPLETE! 🎉           ")
print(f"Best Validation Perplexity: {best_val_ppl:.4f}")
print("="*50 + "\n")

if WANDB_AVAILABLE:
    wandb.finish()

>>> Initializing Final Integrated Training Cell...


wandb: Currently logged in as: heed1193 (ai-m-crown) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ W&B logging is enabled for run: cbe_fgc_20250927-010250
>>> Preparing for main training loop...
- No checkpoint found. Starting training from scratch.
>>> PyTorch 2.0+ detected. Compiling the model for speedup...


/tmp/ipython-input-2176553584.py:83: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=config.train.use_amp)


✅ Model compiled successfully.

             STARTING TRAINING             



Training Epoch 1:   0%|          | 0/8177 [00:00<?, ?it/s]

/content/drive/MyDrive/CBE-FGC-Project/trainer/trainer.py:95: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=scaler.is_enabled()):
/usr/local/lib/python3.12/dist-packages/torch/_inductor/lowering.py:1890: UserWarning: Torchinductor does not support code generation for complex operators. Performance may be worse than eager.
  warnings.warn(
W0927 01:16:56.837000 19291 torch/_inductor/utils.py:1436] [0/0] Not enough SMs to use max_autotune_gemm mode
